# Solutions · 08 · Branching processes

Worked solutions to the exercises of [notebook 08](../notebooks/08_branching_processes.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engstoch is missing): install it from GitHub
    import engstoch
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engstoch"], check=True)
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize
from engstoch import walks as wk
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

R = np.random.default_rng      # R(seed): a fresh, reproducible generator

## Exercise 1

A new virus has offspring distribution negative binomial with mean R0 = 2.5 and dispersion k (variance R0 + R0^2/k). Compute the probability that a single introduction dies out for k = 0.1, 0.5, 1, 10 and infinity (Poisson).

In [2]:
R0 = 2.5
rows = []
for k in (0.1, 0.5, 1.0, 10.0, np.inf):
    if np.isinf(k):
        G = lambda s: np.exp(R0 * (s - 1))
    else:
        G = lambda s, k=k: (1 + R0 / k * (1 - s)) ** (-k)
    q = optimize.brentq(lambda s: G(s) - s, 0, 1 - 1e-12) if G(0.0) < 1 else 1.0
    rows.append((k, q, 1 - q))
print(pd.DataFrame(rows, columns=["dispersion k", "P(extinct)", "P(major outbreak)"]).to_string(index=False, float_format="%.4f"))

 dispersion k  P(extinct)  P(major outbreak)
       0.1000      0.8607             0.1393
       0.5000      0.5583             0.4417
       1.0000      0.4000             0.6000
      10.0000      0.1437             0.8563
          inf      0.1074             0.8926


The negative binomial pgf is (1 + (R₀/k)(1 − s))^{−k}. With strong overdispersion (k = 0.1, as for SARS) most
infected people infect nobody and a few infect many: a single introduction dies out 86 % of the
time even with R₀ = 2.5, whereas Poisson offspring gives only 11 %. Superspreading makes outbreaks rarer but
explosive when they happen.

## Exercise 2

For Poisson(1.0) offspring (critical), estimate P(Z_n > 0) for n = 10..1000 by iterating the pgf and show that it behaves like 2/(sigma^2 n) (Kolmogorov's estimate).

In [3]:
G = wk.pgf_poisson(1.0)
qn = wk.extinction_by_generation(G, 1000)
rows = [(n, 1 - qn[n], 2 / n, (1 - qn[n]) * n / 2) for n in (10, 30, 100, 300, 1000)]
print(pd.DataFrame(rows, columns=["n", "P(Z_n > 0)", "2/(sigma^2 n)", "ratio"]).to_string(index=False, float_format="%.5f"))

   n  P(Z_n > 0)  2/(sigma^2 n)   ratio
  10     0.15824        0.20000 0.79118
  30     0.06068        0.06667 0.91015
 100     0.01935        0.02000 0.96764
 300     0.00659        0.00667 0.98781
1000     0.00199        0.00200 0.99591


At criticality the survival probability decays only like 2/(σ²n) (Kolmogorov 1938; σ² = 1 for Poisson(1)),
with a ratio that tends to 1 slowly - the correction is of order log n / n. Critical populations die out
surely but slowly, and those still alive at generation n have size of order n.

## Exercise 3

**Implement it yourself:** simulate the total progeny of a subcritical Poisson(0.8) process 10^4 times and compare its distribution with the Borel law P(T = n) = e^{-mn} (mn)^{n-1} / n!.

In [4]:
m = 0.8
tot = []
for i in range(10_000):
    z = wk.simulate_galton_watson(lambda k, r: r.poisson(m, k), 200, R(i))
    tot.append(z.sum())
tot = np.array(tot)
ns = np.arange(1, 16)
borel = np.exp(-m * ns) * (m * ns) ** (ns - 1) / np.array([math.factorial(int(k)) for k in ns])
emp = np.array([np.mean(tot == k) for k in ns])
print(pd.DataFrame({"n": ns, "simulated": emp.round(4), "Borel": borel.round(4)}).T.to_string(header=False))
print(f"mean total progeny {tot.mean():.3f} vs 1/(1 - m) = {wk.total_progeny_mean(wk.pgf_poisson(m)):.3f}")

n          1.0000  2.0000  3.0000  4.0000  5.0000  6.0000  7.0000  8.0000  9.0000  10.0000  11.0000  12.0000  13.0000  14.0000  15.0000
simulated  0.4574  0.1491  0.0860  0.0557  0.0390  0.0285  0.0262  0.0184  0.0146   0.0125   0.0104   0.0088   0.0075   0.0075   0.0049
Borel      0.4493  0.1615  0.0871  0.0557  0.0391  0.0291  0.0226  0.0181  0.0149   0.0124   0.0105   0.0090   0.0078   0.0068   0.0060
mean total progeny 5.057 vs 1/(1 - m) = 5.000


The total progeny of a Poisson(m) branching process is Borel-distributed - a consequence of the hitting-time
theorem for the random walk that explores the family tree. Its mean 1/(1 − m) = 5 hides a long tail: one
family in a hundred exceeds 40.